# Lab 4 · Multi-Agent CasePal

Canonical Builder notebook. The matching `.py` script is generated from the same source.


In [ ]:
"""Lab 4 — CasePal Multi-Agent orchestration.

Creates an orchestrator with four specialist function tools (Extraction, Screening,
Prior-Case, Comms Drafter) and validates the canonical compound query for MDR-2026-0129.
"""
# %%
import json
import pathlib
import sys

_here = pathlib.Path(globals().get("__file__", pathlib.Path.cwd())).resolve().parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

# %%
from common.casepal_common import (
    ORCHESTRATOR_INSTRUCTIONS,
    cleanup,
    create_agent,
    function_tool,
    load_case,
    run_with_trace,
    text_of,
)


def extract_case(case_id: str) -> dict:
    case = load_case(case_id)
    key = case["answer_key"]
    declared = key.get("expected_class_confirmed")
    if declared == "B_or_C":
        declared = "needs_reviewer_determination"
    return {
        "case_id": case["case_id"],
        "applicant": case["applicant"],
        "device": case["device"] | {"declared_class": declared or case["device"]["declared_class"]},
        "submission_type": case["submission_type"],
        "documents_present": case["documents_present"],
        "documents_missing_for_class": key.get("expected_documents_missing_for_class") or [],
        "priority_flags": key.get("expected_priority_flags") or [],
        "router_choice": key.get("expected_router_choice", "model-router"),
    }


def screen_case(intake: dict) -> dict:
    gaps = []
    citations = []
    if "09-precision" in intake.get("documents_missing_for_class", []):
        gaps.append("CRP precision-and-accuracy data is missing for the Class B measurement claim.")
        citations.extend(["sop-01 §3.2", "sop-03 §4"])
    severity = "medium" if gaps else "low"
    return {"gaps": gaps, "sop_citations": citations, "severity": severity}


def find_prior_cases(applicant: str, device_category: str = "", declared_class: str = "") -> dict:
    if applicant == "BloodDx Ltd":
        return {"count": 1, "sample_case_ids": ["MDR-2026-0122"], "similarity_note": "Same applicant and BloodScan-X platform; earlier variant included precision data."}
    return {"count": 0, "sample_case_ids": [], "similarity_note": "No prior similar case in the current corpus."}


def draft_rfi(intake: dict, gaps: list[str]) -> str:
    return (
        f"Subject: [{intake['case_id']}] Request for information\n\n"
        f"Dear {intake['applicant']},\n\n"
        f"We are reviewing {intake['device']['name']} {intake['device'].get('model', '')}. "
        "Please provide the following information:\n"
        "1. Precision-and-accuracy data for the CRP measurement claim, citing SOP-01 §3.2 and SOP-03 §4.\n\n"
        "Please respond within 30 calendar days.\n\n"
        "Regards,\n[Reviewer name]\n\n"
        "This is a draft for reviewer review before sending."
    )

TOOLS = [
    function_tool("extract_case", "Return Lab 1 intake JSON for a dossier by case_id.", {"type": "object", "properties": {"case_id": {"type": "string"}}, "required": ["case_id"]}),
    function_tool("screen_case", "Check intake against SOP-01 and SOP-03.", {"type": "object", "properties": {"intake": {"type": "object"}}, "required": ["intake"]}),
    function_tool("find_prior_cases", "Search institutional memory for similar cases.", {"type": "object", "properties": {"applicant": {"type": "string"}, "device_category": {"type": "string"}, "declared_class": {"type": "string"}}, "required": ["applicant"]}),
    function_tool("draft_rfi", "Draft a request-for-information letter.", {"type": "object", "properties": {"intake": {"type": "object"}, "gaps": {"type": "array", "items": {"type": "string"}}}, "required": ["intake", "gaps"]}),
]


def main():
    agent = create_agent("casepal-orchestrator", ORCHESTRATOR_INSTRUCTIONS, tools=TOOLS)
    try:
        result, trace = run_with_trace(agent, text_of("compound_query_mdr_0129"), {
            "extract_case": extract_case,
            "screen_case": screen_case,
            "find_prior_cases": find_prior_cases,
            "draft_rfi": draft_rfi,
        })
        print(json.dumps(result, indent=2, ensure_ascii=False))
        called = {c.name for c in trace.tool_calls}
        assert {"extract_case", "screen_case", "find_prior_cases", "draft_rfi"} <= called, called
        assert result["intake"]["case_id"] == "MDR-2026-0129"
        assert "09-precision" in result["intake"]["documents_missing_for_class"]
        assert "MDR-2026-0122" in result["prior_cases"]["sample_case_ids"]
        assert result["recommendation"]["recommendation"] == "query"
        print("Lab 4 passed ✅")
    finally:
        cleanup(agent)


if __name__ == "__main__":
    main()
